In [20]:
import requests, warnings
import pandas as pd
from google.colab import files
from io import StringIO
from time import sleep

In [26]:
ids = pd.read_csv(
    StringIO('''
72520514762
72219013874
07481099999
72534014819
76679399999
76679399999
72572024127
72494523293
47113099999
47113099999
74486094789
07157099999
72422093820
72640014839
72658014922
83775399999
07157099999
72509014739
47113099999
0618009999
0618009999
94767099999
'''),
    header=None,
    dtype=str
)

In [27]:
ids

,0
0,72520514762
1,72219013874
2,07481099999
3,72534014819
4,76679399999
5,76679399999
6,72572024127
7,72494523293
8,47113099999
9,47113099999


In [28]:
ids.columns = ["station_id"]

BASE_URL = "https://www.ncei.noaa.gov/access/services/search/v1/autocomplete"

results = []

In [29]:
ids

,station_id
0,72520514762
1,72219013874
2,07481099999
3,72534014819
4,76679399999
5,76679399999
6,72572024127
7,72494523293
8,47113099999
9,47113099999


In [30]:
for station_id in ids["station_id"].astype(str):
    params = {
        "field": "stations",
        "dataset": "global-hourly",
        "text": station_id
    }

    try:
        r = requests.get(BASE_URL, params=params, timeout=10)
        r.raise_for_status()

        data = r.json()

        is_valid = len(data.get("results", [])) > 0

        # Optional: grab station name if valid
        station_name = (
            data["results"][0]["name"]
            if is_valid else None
        )

        results.append({
            "station_id": station_id,
            "valid": is_valid,
            "station_name": station_name
        })

    except Exception as e:
        results.append({
            "station_id": station_id,
            "valid": False,
            "station_name": None,
            "error": str(e)
        })

    # polite pause to avoid hammering API
    sleep(0.1)

In [31]:
# Convert to DataFrame
results_df = pd.DataFrame(results)

print(results_df)

     station_id  valid                                       station_name
0   72520514762   True             PITTSBURGH ALLEGHENY CO AIRPORT, PA US
1   72219013874   True  ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPO...
2   07481099999   True                             LYON SAINT EXUPERY, FR
3   72534014819   True                      CHICAGO MIDWAY AIRPORT, IL US
4   76679399999   True  LICENCIADO BENITO JUAREZ INTERNATIONAL MEXICO ...
5   76679399999   True  LICENCIADO BENITO JUAREZ INTERNATIONAL MEXICO ...
6   72572024127   True        SALT LAKE CITY INTERNATIONAL AIRPORT, UT US
7   72494523293   True                                    SAN JOSE, CA US
8   47113099999   True                  INCHEON INTERNATIONAL AIRPORT, KS
9   47113099999   True                  INCHEON INTERNATIONAL AIRPORT, KS
10  74486094789   True                   JFK INTERNATIONAL AIRPORT, NY US
11  07157099999   True                              CHARLES DE GAULLE, FR
12  72422093820   True                

In [32]:
def fetch_top_left(station_id):
    url = (
        "https://www.ncei.noaa.gov/access/services/search/v1/data"
        f"?dataset=global-hourly&stations={station_id}"
    )

    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()

        top_left = data.get("bounds", {}).get("topLeft", {})

        return pd.Series({
            "top_left_lat": top_left.get("lat"),
            "top_left_lon": top_left.get("lon")
        })

    except Exception:
        return pd.Series({
            "top_left_lat": None,
            "top_left_lon": None
        })

In [33]:
# Add coordinates to dataframe
coords = ids["station_id"].apply(fetch_top_left)

df = pd.concat([ids, coords], axis=1)

     station_id  top_left_lat  top_left_lon
0   72520514762     40.355100    -79.921670
1   72219013874     33.630100    -84.442240
2   07481099999     45.726387      5.090833
3   72534014819     41.786110    -87.755140
4   76679399999     19.436303    -99.072097
5   76679399999     19.436303    -99.072097
6   72572024127     40.778100   -111.969400
7   72494523293     37.359380   -121.924440
8   47113099999     37.466667    126.433333
9   47113099999     37.466667    126.433333
10  74486094789     40.639150    -73.764010
11  07157099999     49.012779      2.550000
12  72422093820     38.040800    -84.611380
13  72640014839     42.955000    -87.904570
14  72658014922     44.885230    -93.231330
15  83775399999    -23.433000    -46.467000
16  07157099999     49.012779      2.550000
17  72509014739     42.360600    -71.009750
18  47113099999     37.466667    126.433333
19   0618009999           NaN           NaN
20   0618009999           NaN           NaN
21  94767099999    -33.946111   

In [34]:
print(df.to_csv(sep='\t', index=False))

station_id	top_left_lat	top_left_lon
72520514762	40.35509996116161	-79.92167003452778
72219013874	33.630099976435304	-84.44224004633725
07481099999	45.726386960595846	5.090832961723208
72534014819	41.78610997740179	-87.75514000095427
76679399999	19.436302995309234	-99.07209707424045
76679399999	19.436302995309234	-99.07209707424045
72572024127	40.778099996969104	-111.96940004825592
72494523293	37.35937998164445	-121.92444001324475
47113099999	37.46666658204049	126.43333327956498
47113099999	37.46666658204049	126.43333327956498
74486094789	40.639149961061776	-73.76401006244123
07157099999	49.01277897879481	2.549999998882413
72422093820	38.04079995956272	-84.61138006299734
72640014839	42.95499998610467	-87.90457005612552
72658014922	44.8852299945429	-93.23133003897965
83775399999	-23.433000035583973	-46.467000022530556
07157099999	49.01277897879481	2.549999998882413
72509014739	42.360599962994456	-71.00975004024804
47113099999	37.46666658204049	126.43333327956498
0618009999		
0618009999	

In [35]:
BASE_URL = "https://www.ncei.noaa.gov/access/services/data/v1"

start = 2015
end = 2020

results = []

In [36]:
for station in df["station_id"].astype(str):

    params = {
        "dataset": "global-historical-climatology-network-hourly",
        "format": "json",  # easier for checking validity
        "units": "metric",
        "dataTypes": (
            "STATION,Year,Month,Day,Hour,"
            "temperature,relative_humidity"
        ),
        "stations": station,
        "startDate": f"{start - 5}-01-01",
        "endDate": f"{end + 5}-12-31"
    }

    try:
        r = requests.get(BASE_URL, params=params, timeout=30)

        status_code = r.status_code

        # Sometimes NOAA returns 200 with empty payload
        has_data = False
        row_count = 0
        error = None

        if status_code == 200:

            # Try JSON first
            try:
                data = r.json()

                if isinstance(data, list) and len(data) > 0:
                    has_data = True
                    row_count = len(data)

            except Exception:
                # fallback if response isn't valid JSON
                text = r.text.strip()

                # NOAA sometimes returns CSV text
                if len(text) > 0 and "STATION" in text:
                    has_data = True
                    row_count = len(text.splitlines()) - 1

        else:
            error = r.text[:300]

        results.append({
            "station_id": station,
            "status_code": status_code,
            "has_data": has_data,
            "row_count": row_count,
            "error": error
        })

    except Exception as e:
        results.append({
            "station_id": station,
            "status_code": None,
            "has_data": False,
            "row_count": 0,
            "error": str(e)
        })

    sleep(0.2)  # avoid hammering NOAA

In [37]:
# Results dataframe
api_test_df = pd.DataFrame(results)

print(api_test_df)

# Successful stations
working = api_test_df[api_test_df["has_data"]]

# Failed stations
failed = api_test_df[~api_test_df["has_data"]]

print("\nWORKING STATIONS")
print(working)

print("\nFAILED STATIONS")
print(failed)

     station_id  status_code  has_data  row_count error
0   72520514762          200     False          0  None
1   72219013874          200     False          0  None
2   07481099999          200     False          0  None
3   72534014819          200     False          0  None
4   76679399999          200     False          0  None
5   76679399999          200     False          0  None
6   72572024127          200     False          0  None
7   72494523293          200     False          0  None
8   47113099999          200     False          0  None
9   47113099999          200     False          0  None
10  74486094789          200     False          0  None
11  07157099999          200     False          0  None
12  72422093820          200     False          0  None
13  72640014839          200     False          0  None
14  72658014922          200     False          0  None
15  83775399999          200     False          0  None
16  07157099999          200     False          